In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CustomerAnalysis") \
    .master("local[2]") \
    .getOrCreate()

print("SparkSession created successfully...!!!")

SparkSession created successfully...!!!


In [4]:
customers_df = spark.read.csv(
    "customers_dataset.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "products_dataset.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "orders_dataset.csv",
    header=True,
    inferSchema=True
)

In [5]:
customers_df.printSchema()
products_df.printSchema()
orders_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- membership_type: string (nullable = true)
 |-- credit_score: integer (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- stock_quantity: integer (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percentage: integer (nullable = true)
 |-- payment_mode: str

In [6]:
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")
orders_df.createOrReplaceTempView("orders")

In [7]:
spark.sql("SELECT * FROM customers").show(5)
spark.sql("SELECT * FROM products").show(5)
spark.sql("SELECT * FROM orders").show(5)

+-----------+---------------+------+---+---------+-----------+-----------+---------------+------------+
|customer_id|  customer_name|gender|age|     city|      state|signup_date|membership_type|credit_score|
+-----------+---------------+------+---+---------+-----------+-----------+---------------+------------+
|          1|Zachary Randall|  Male| 22|     Pune|Maharashtra| 2023-09-03|         Silver|         614|
|          2|  Morgan Wilson|  Male| 27|     Pune|  Telangana| 2025-09-26|         Silver|         802|
|          3|     Troy Brown|Female| 23|  Chennai| Tamil Nadu| 2026-03-27|         Silver|         619|
|          4| Crystal Miller|  Male| 56|Bangalore|  Telangana| 2025-07-29|           Gold|         612|
|          5|  Marisa Miller|Female| 58|   Mumbai| Tamil Nadu| 2025-07-17|         Silver|         857|
+-----------+---------------+------+---+---------+-----------+-----------+---------------+------------+
only showing top 5 rows

+----------+----------------+----------

In [8]:
spark.sql("""SELECT category, AVG(price) AS avg_product_price FROM products GROUP BY category""").show()

+-----------+------------------+
|   category| avg_product_price|
+-----------+------------------+
|    Fashion|           17298.6|
|     Sports|          27627.75|
|    Grocery|           26990.0|
|Electronics|           28045.0|
|  Furniture|30423.176470588234|
+-----------+------------------+



In [11]:
spark.sql("""SELECT
        o.order_id,
        c.customer_name,
        p.product_name,
        o.quantity,
        o.sales_amount
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
""").show()

+--------+---------------+----------------+--------+------------------+
|order_id|  customer_name|    product_name|quantity|      sales_amount|
+--------+---------------+----------------+--------+------------------+
|       1|   Jason Nelson|   Visit Product|       4|           22028.6|
|       2|    Nicole Meza|    Fill Product|       3|          64296.18|
|       3| Emily Galloway|  Assume Product|       3|46010.700000000004|
|       4|     John Rivas|    Week Product|       1|10746.119999999999|
|       5|  Joseph Watson|    Fill Product|       3|         105099.84|
|       6|   Jordan Jones|    Base Product|       5|           63109.8|
|       7|     Ryan Cohen|    Name Product|       2|13648.199999999999|
|       8|  Mitchell Kent|    Best Product|       3|          61558.38|
|       9|     Gina Ortiz|    Wear Product|       2|           66830.4|
|      10|   Shane Miller| Popular Product|       3|           93219.0|
|      11|   Jason Nelson| Soldier Product|       4|          12

In [12]:
spark.sql("""SELECT
        product_id,
        product_name,
        category,
        price
    FROM products
    ORDER BY price DESC
    LIMIT 5
""").show()

+----------+--------------+-----------+-----+
|product_id|  product_name|   category|price|
+----------+--------------+-----------+-----+
|         9|  Cold Product|  Furniture|49683|
|         7| Place Product|  Furniture|48408|
|        15|Return Product|Electronics|47690|
|        29|   Mrs Product|    Grocery|46900|
|        31|  Best Product|     Sports|46205|
+----------+--------------+-----------+-----+



In [13]:
spark.sql("""
    SELECT p.category, SUM(o.sales_amount) AS total_revenue FROM orders o
    JOIN products p ON o.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
""").show()

+-----------+--------------------+
|   category|       total_revenue|
+-----------+--------------------+
|  Furniture|1.0324294070000004E7|
|    Grocery|   7728527.300000001|
|    Fashion|          5952276.32|
|Electronics|   5205559.010000001|
|     Sports|  3847011.2800000003|
+-----------+--------------------+



In [14]:
spark.sql("""
    select c.membership_type, COUNT(o.order_id) AS number_of_orders FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.membership_type
    ORDER BY number_of_orders DESC
""").show()

+---------------+----------------+
|membership_type|number_of_orders|
+---------------+----------------+
|         Silver|             200|
|       Platinum|             162|
|           Gold|             138|
+---------------+----------------+



In [ ]:
spark.sql("""select c.* FROM customers c LEFT ANTI JOIN orders o ON c.customer_id = o.customer_id""").show()

In [ ]:
spark.sql("""
    select c.* FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NULL
""").show()

+-----------+-------------+------+---+----+----------+-----------+---------------+------------+
|customer_id|customer_name|gender|age|city|     state|signup_date|membership_type|credit_score|
+-----------+-------------+------+---+----+----------+-----------+---------------+------------+
|         37|   Robin Cobb|  Male| 25|Pune|Tamil Nadu| 2024-09-10|         Silver|         534|
+-----------+-------------+------+---+----+----------+-----------+---------------+------------+



In [16]:
spark.sql("""SELECT
        c.city,
        COUNT(o.order_id) AS number_of_orders,
        SUM(o.sales_amount) AS total_spending
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.city
    ORDER BY total_spending DESC
""").show()

+---------+----------------+------------------+
|     city|number_of_orders|    total_spending|
+---------+----------------+------------------+
|  Chennai|              92| 6431761.559999998|
|    Delhi|              88| 6172928.729999999|
|Bangalore|              86| 6147224.900000004|
|   Mumbai|              85|        5405864.12|
|     Pune|              83|        4735115.74|
|Hyderabad|              66|4164772.9300000006|
+---------+----------------+------------------+



In [19]:
spark.sql("""
SELECT city, SUM(ROUND(sales_amount, 2)) AS total_sales,
DENSE_RANK() OVER(ORDER BY SUM(ROUND(sales_amount, 2)) DESC) AS rnk
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY city
""").filter("rnk <= 3").show()

+---------+-----------------+---+
|     city|      total_sales|rnk|
+---------+-----------------+---+
|  Chennai|6431761.559999999|  1|
|    Delhi|6172928.729999999|  2|
|Bangalore|6147224.900000004|  3|
+---------+-----------------+---+



In [21]:
spark.sql("""Select concat(customer_name, ' - ', city) as customer_details from customers""").show(20, truncate=False)

+--------------------------+
|customer_details          |
+--------------------------+
|Zachary Randall - Pune    |
|Morgan Wilson - Pune      |
|Troy Brown - Chennai      |
|Crystal Miller - Bangalore|
|Marisa Miller - Mumbai    |
|Theresa Ferguson - Mumbai |
|Yvonne Carroll - Chennai  |
|Michael Sloan - Mumbai    |
|Mark Thompson - Chennai   |
|Tara Walker - Bangalore   |
|Darrell Jones - Chennai   |
|Anthony Delgado - Pune    |
|William Bailey - Pune     |
|Lisa Lopez DDS - Pune     |
|Emily Galloway - Delhi    |
|Ryan Cohen - Chennai      |
|Jacob Lopez - Chennai     |
|Jaime Vasquez - Pune      |
|Hector Gordon - Mumbai    |
|Paul Stevens - Delhi      |
+--------------------------+
only showing top 20 rows



In [22]:
spark.sql("""
SELECT category, product_name, total_sales
FROM (
    SELECT p.category, p.product_name,
           SUM(o.sales_amount) AS total_sales,
           DENSE_RANK() OVER (
               PARTITION BY p.category
               ORDER BY SUM(o.sales_amount) DESC
           ) AS rnk
    FROM products p
    JOIN orders o ON p.product_id = o.product_id
    GROUP BY p.category, p.product_name
)
WHERE rnk <= 3
""").show()

+-----------+---------------+------------------+
|   category|   product_name|       total_sales|
+-----------+---------------+------------------+
|Electronics| Worker Product| 742170.8500000001|
|Electronics| Return Product| 696393.4400000001|
|Electronics|Section Product|         652906.61|
|    Fashion|  Build Product|         1215711.4|
|    Fashion|   Door Product|         916884.73|
|    Fashion|Brother Product|         722775.46|
|  Furniture|  Place Product|        1193462.47|
|  Furniture|  Plant Product|         912991.26|
|  Furniture| Listen Product| 882490.3800000001|
|    Grocery|   Fill Product| 763266.6300000001|
|    Grocery|    Car Product|         748916.44|
|    Grocery|    Say Product| 704060.6300000001|
|     Sports|  Visit Product| 749711.3200000001|
|     Sports|   Name Product| 703489.3300000001|
|     Sports|  Cover Product|515701.30000000005|
+-----------+---------------+------------------+



In [23]:
spark.sql("""
SELECT city, customer_name, total_spending
FROM (
    SELECT c.city, c.customer_name,
           SUM(o.sales_amount) AS total_spending,
           ROW_NUMBER() OVER (
               PARTITION BY c.city
               ORDER BY SUM(o.sales_amount) DESC
           ) AS rn

    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.city, c.customer_name
)
WHERE rn = 1""").show()

+---------+----------------+-----------------+
|     city|   customer_name|   total_spending|
+---------+----------------+-----------------+
|Bangalore|    Shane Miller|         581160.2|
|  Chennai|   Darrell Jones|592001.0900000001|
|    Delhi|Shirley Harrison|        845556.64|
|Hyderabad|    Jason Nelson|        591651.49|
|   Mumbai|Theresa Ferguson|        720304.25|
|     Pune|   Kelly Jackson|         411116.0|
+---------+----------------+-----------------+



In [9]:
spark.sql("""
    SELECT
        c.customer_id,
        c.customer_name,
        o.order_id,
        p.product_name,
        p.category,
        o.quantity,
        o.sales_amount
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
""").show()

+-----------+---------------+--------+----------------+-----------+--------+------------------+
|customer_id|  customer_name|order_id|    product_name|   category|quantity|      sales_amount|
+-----------+---------------+--------+----------------+-----------+--------+------------------+
|         45|   Jason Nelson|       1|   Visit Product|     Sports|       4|           22028.6|
|         64|    Nicole Meza|       2|    Fill Product|    Grocery|       3|          64296.18|
|         15| Emily Galloway|       3|  Assume Product|  Furniture|       3|46010.700000000004|
|         47|     John Rivas|       4|    Week Product|     Sports|       1|10746.119999999999|
|        119|  Joseph Watson|       5|    Fill Product|    Grocery|       3|         105099.84|
|         21|   Jordan Jones|       6|    Base Product|  Furniture|       5|           63109.8|
|         16|     Ryan Cohen|       7|    Name Product|     Sports|       2|13648.199999999999|
|         62|  Mitchell Kent|       8|  